# Hotel Booking with Priority Member Middleware

This notebook demonstrates **function-based middleware** using the Microsoft Agent Framework. We build upon the conditional workflow example by adding a middleware layer that gives priority members special privileges.

## What You'll Learn:
1. **Function-Based Middleware**: Intercept and modify function results
2. **Context Access**: Read and modify `context.result` after execution
3. **Business Logic Implementation**: Priority member benefits
4. **Result Override**: Change function outcomes based on user status
5. **Same Workflow, Different Outcomes**: Middleware-driven behavior changes

## Workflow Architecture with Middleware:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Key Difference from Conditional Workflow:

**Without Middleware** (14-conditional-workflow.ipynb):
- Paris has no rooms → Route to alternative_agent

**With Middleware** (this notebook):
- Regular user + Paris → No rooms → Route to alternative_agent
- Priority user + Paris → 🌟 Middleware overrides! → Available → Route to booking_agent

## Prerequisites:
- Microsoft Agent Framework installed
- Understanding of conditional workflows (see 14-conditional-workflow.ipynb)
- GitHub token or OpenAI API key
- Basic understanding of middleware patterns


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## 1 žingsnis: apibrėžkite Pydantic modelius struktūruotiems rezultatams

Šie modeliai apibrėžia **schemą**, kurią agentai grąžins. Mes pridėjome lauką `priority_override`, kad būtų galima sekti, kai tarpinė programa keičia prieinamumo rezultatą.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## 2 žingsnis: Apibrėžkite Prioritetinių Narių Duomenų Bazę

Šiai demonstracijai mes simuliuosime prioritetinių narių duomenų bazę. Veikloje tai būtų kreipimasis į tikrą duomenų bazę arba API.

**Prioritetiniai nariai:**
- `alice@example.com` - VIP narys
- `bob@example.com` - Premium narys  
- `priority_user` - Testo paskyra


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## 3 veiksmas: Sukurkite viešbučio užsakymo įrankį

Tas pats kaip sąlyginis darbo procesas, bet dabar jis bus perimtas tarpinio programavimo sluoksnio!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## 4 veiksmas: 🌟 Sukurkite prioritetinės tikrinimo tarpinį sluoksnį (SVARBIAUSIA FUNKCIJA!)

Tai yra šio užrašo **pagrindinė funkcija**. Tarpinis sluoksnis:

1. **Perimta** hotel_booking funkcijos kvietimas
2. **Atlieka** funkciją įprastai kviesdamas `next(context)`
3. **Tiria** rezultatą `context.result`
4. **Pakeičia** rezultatą, jei vartotojas yra prioritetinis ir nėra kambarių
5. **Grąžina** pakeistą rezultatą atgal agentui

**Pagrindinis modelis:**
```python
async def my_middleware(context, next):
    await next(context)  # Vykdyti funkciją
    # Dabar context.result turi funkcijos rezultatą
    if some_condition:
        context.result = new_value  # Pakeisti!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## 5 veiksmas: Apibrėžkite sąlygines funkcijas maršruto nustatymui

Tos pačios sąlyginės funkcijos kaip sąlyginiame darbo eigoje – jos tikrina struktūruotą išvestį, kad nustatytų maršrutą.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## 6 žingsnis: Sukurkite pasirinktą rodymo vykdytoją

Tas pats vykdytojas kaip ir anksčiau – rodo galutinį darbo eigos rezultatą.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## 7 žingsnis: Įkelkite aplinkos kintamuosius

Konfigūruokite LLM klientą (GitHub modeliai arba OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## 8 žingsnis: Sukurkite AI agentus su tarpinio programinės įrangos sluoksniu

**PAGRINDINIS SKIRTUMAS:** Kai kuriame availability_agent, mes perduodame `middleware` parametrą!

Taip mes įtraukiame priority_check_middleware į agente funkcijų kvietimo vamzdyną.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## 9 žingsnis: Sukurkite darbo eigą

Ta pati darbo eigos struktūra kaip ir anksčiau – sąlyginis maršrutizavimas pagal pasiekiamumą.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## 10 žingsnis: Testo atvejis 1 - Įprastas vartotojas Paryžiuje (Be perrašymo)

Įprastas vartotojas bando užsakyti Paryžių → Nėra kambarių → Nukreipia į alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## 11 žingsnis: 2 testas - 🌟 Prioritetinis vartotojas Paryžiuje (SU perrašymu!)

Prioritetinis narys bando rezervuoti Paryžių → Iš pradžių nėra kambarių → 🌟 Tarpinis sluoksnis perrašo! → Nukreipiama į booking_agent

**Tai pagrindinis tarpinių sluoksnių galios demonstravimas!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## 12 žingsnis: 3 testavimo atvejis - Prioritetinis naudotojas Stokholme (jau pasiekiamas)

Prioritetinis naudotojas bando Stokholmą → Kambariai yra → Nereikia perrašyti → Nukreipia į booking_agent

Tai parodo, kad tarpinis programinis sluoksnis veikia tik tada, kai reikia!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## Pagrindinės išvados ir tarpinės programos sąvokos

### ✅ Ko išmokote:

#### **1. Funkcijomis pagrįstas tarpinės programos modelis**

Tarpinė programa perima funkcijų kvietimus naudodama paprastą asinchroninę funkciją:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # Prieš funkcijos vykdymą
    print("Intercepting...")
    
    # Vykdyti funkciją
    await next(context)
    
    # Po funkcijos vykdymo - patikrinti rezultatą
    if context.result:
        # Prireikus pakeisti rezultatą
        context.result = modified_value
```

#### **2. Konteksto prieiga ir rezultato pakeitimas**

- `context.function` - Prieiga prie kviečiamos funkcijos
- `context.arguments` - Funkcijos argumentų skaitymas
- `context.kwargs` - Papildomų parametrų prieiga
- `await next(context)` - Funkcijos vykdymas
- `context.result` - Funkcijos rezultato skaitymas / keitimas

#### **3. Verslo logikos įgyvendinimas**

Mūsų tarpinė programa įgyvendina prioritetines narių naudą:
- **Įprasti naudotojai**: Jokių pakeitimų, standartinis darbo eiga
- **Prioritetiniai naudotojai**: Keičia „nėra prieinamumo“ į „prieinamas“
- **Sąlyginė logika**: Pakeitimus daro tik kai reikia

#### **4. Ta pati darbo eiga, skirtingi rezultatai**

Tarpinės programos galia:
- ✅ Darbo eigos struktūra nesikeičia
- ✅ Įrankio funkcija nesikeičia
- ✅ Sąlyginės maršruto logikos nesikeičia
- ✅ Tiesiog tarpinė programa → Skirtingas elgesys!

### 🚀 Realūs pritaikymai:

1. **VIP/Premium funkcijos**
   - Pakeičiamos ribos premium naudotojams
   - Teikiama prioritetinė prieiga prie išteklių
   - Dinamiškai atrakina premium funkcijas

2. **A/B testavimas**
   - Veda naudotojus į skirtingus įgyvendinimus
   - Testuoja naujas funkcijas su tam tikrais naudotojais
   - Laipsniški funkcijų išleidimai

3. **Saugumas ir atitiktis**
   - Audituoja funkcijų kvietimus
   - Blokuoja jautrias operacijas
   - Užtikrina verslo taisyklių laikymąsi

4. **Veikimo optimizavimas**
   - Tiesioginė rezultatų kešavimo funkcija tam tikriems naudotojams
   - Vengia brangių operacijų kai įmanoma
   - Dinaminis išteklių paskirstymas

5. **Klaidų valdymas ir pakartotinis bandymas**
   - Gražiai pagauna ir apdoroja klaidas
   - Įgyvendina pakartotinio bandymo logiką
   - Atsarginiai sprendimai alternatyviems įgyvendinimams

6. **Registravimas ir stebėjimas**
   - Stebi funkcijų vykdymo laiką
   - Registruoja parametrus ir rezultatus
   - Stebi naudojimo modelius

### 🔑 Pagrindiniai skirtumai nuo dekoratorių:

| Funkcija | Dekoratorius | Tarpinė programa |
|---------|-----------|------------|
| **Apimtis** | Viena funkcija | Visos agento funkcijos |
| **Lankstumas** | Fiksuotas apibrėžimo metu | Dinamiškas vykdymo metu |
| **Kontekstas** | Ribotas | Pilnas agento kontekstas |
| **Sudėtis** | Keli dekoratoriai | Tarpinių programų vamzdis |
| **Agentui žinoma** | Ne | Taip (prieiga prie agento būsenos) |

### 📚 Kada naudoti tarpinę programą:

✅ **Naudokite tarpinę programą, kai:**
- Reikia keisti elgesį pagal naudotojo / sesijos būseną
- Norite pritaikyti logiką kelioms funkcijoms
- Reikia prieigos prie agento lygmens konteksto
- Įgyvendinate skerspjūvio užduotis (registravimas, autentifikavimas ir kt.)

❌ **Nenaudokite tarpinės programos, kai:**
- Paprastas įvesties validavimas (naudokite Pydantic)
- Funkcijai būdinga logika (laikykite funkcijoje)
- Vienkartiniai pakeitimai (tiesiog pakeiskite funkciją)

### 🎓 Pažangūs modeliai:

```python
# Kelios tarpinės programos (vykdymo tvarka svarbi!)
middleware=[
    logging_middleware,      # Užrašai pirmiausia
    auth_middleware,         # Tada tikrina autentifikaciją
    cache_middleware,        # Tada tikrina talpyklą
    rate_limit_middleware,   # Tada riboja užklausų dažnį
    priority_check_middleware  # Galiausiai prioritetų tikrinimas
]

# Sąlyginis tarpinės programos vykdymas
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # Modifikuoti rezultatą
    else:
        # Visiškai praleisti vykdymą
        context.result = cached_value
```

### 🔗 Susijusios sąvokos:

- **Agentas Tarpinė programa**: Perima agent.run() kvietimus
- **Funkcijų Tarpinė programa**: Perima įrankio funkcijų kvietimus (ką mes naudojome!)
- **Tarpinių programų vamzdis**: Tarpinių programų grandinė, vykdoma paeiliui
- **Konteksto propagacija**: Perduoda būseną per tarpinės programos grandinę


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
